# Phase 6d Wave 1: Center Symmetry, Confinement, and the Walker-Wang KSS Window — Technical Notebook

Companion to `papers/paper36_center_symmetry/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/CenterSymmetryConfinement.lean` (18 substantive theorems / 0 sorry / 0 new axioms; verified `propext, Classical.choice, Quot.sound` only).

**Structure (mirrors paper §1–§7):**
1. ℤ_N center symmetry: `CenterZN`, `centerPhase`, root-of-unity properties
2. Polyakov loop and confinement: `confining_iff_magnitude_zero`, `confining_iff_center_invariant`
3. Concrete SU(2) realization: `centerPhase_Z2_eq_neg_one`
4. Svetitsky-Yaffe universality: `critical_exponent_nu`, `ising_nu_gt_potts_nu`
5. KSS bound: $1/(4\pi) \in [0.07, 0.08]$
6. Walker-Wang transport correctness-push and falsifiers
7. Cross-bridges: `higher_form_discrete_iff_non_abelian`, `su3k1_fusion_card_matches_z3_order`
8. Figure: |P| vs T/T_c with universality-class scaling + KSS / Walker-Wang window
9. Lean theorem inventory

All numerical content imports from `src.center_symmetry` — no inline physics redefinition.

## 1. ℤ_N center symmetry

For SU(N) gauge theory, the center is $\mathbb{Z}_N$ generated by $\zeta_N := \exp(2\pi i / N)$. We require $N \geq 2$ (load-bearing). The Lean structure `CenterZN` carries `N_ge_two` as an invariant.

Properties (Lean):
- `centerPhase_pow_N`: $\zeta_N^N = 1$ (root of unity)
- `centerPhase_norm_one`: $|\zeta_N| = 1$
- `centerPhase_Z2_eq_neg_one`: $\zeta_2 = -1$ (concrete SU(2))

In [1]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.center_symmetry import (
    CenterZN, center_phase, center_action,
    confining, polyakov_magnitude,
    UniversalityClass, svetitsky_yaffe_class, critical_exponent_nu,
    KSS_BOUND, WalkerWangPrediction, walker_wang_consistent_with_kss,
)
from src.core.visualizations import COLORS, fig_polyakov_loop_deconfinement
import math

for N in [2, 3, 4]:
    z = CenterZN(N)
    zeta = center_phase(z)
    zeta_to_N = zeta ** N
    print(f'  Z_{N}: ζ_{N} = {zeta.real:+.6f} + {zeta.imag:+.6f}i    '
          f'|ζ_{N}| = {abs(zeta):.6f}    ζ_{N}^{N} = {zeta_to_N.real:+.6f} + {zeta_to_N.imag:+.6f}i')

  Z_2: ζ_2 = -1.000000 + +0.000000i    |ζ_2| = 1.000000    ζ_2^2 = +1.000000 + -0.000000i
  Z_3: ζ_3 = -0.500000 + +0.866025i    |ζ_3| = 1.000000    ζ_3^3 = +1.000000 + -0.000000i
  Z_4: ζ_4 = +0.000000 + +1.000000i    |ζ_4| = 1.000000    ζ_4^4 = +1.000000 + -0.000000i


## 2. Polyakov loop and confinement

Polyakov loop $P \in \mathbb{C}$ is the order parameter. Confinement = $|\langle P \rangle| = 0$. Two equivalent characterizations:

- **Quantitative:** `confining_iff_magnitude_zero` — $|P| = 0$ iff confining.
- **Symmetric:** `confining_iff_center_invariant` — $P$ is fixed by every center action iff confining.

$P = 0$ is the unique fixed point of non-trivial center action.

In [2]:
Z2 = CenterZN(2)
Z3 = CenterZN(3)
for label, P in [('confining (P = 0)',     0+0j),
                 ('deconfining P = 1',     1+0j),
                 ('deconfining P = 0.5+0.3j', 0.5+0.3j)]:
    P_z2 = center_action(Z2, P)
    P_z3 = center_action(Z3, P)
    print(f'  {label:30s}  |P| = {polyakov_magnitude(P):.4f}  confining = {confining(P)}')
    print(f'    Z_2 acted: {P_z2.real:+.4f}{P_z2.imag:+.4f}j   fixed by Z_2: {abs(P_z2 - P) < 1e-12}')
    print(f'    Z_3 acted: {P_z3.real:+.4f}{P_z3.imag:+.4f}j   fixed by Z_3: {abs(P_z3 - P) < 1e-12}')
    print()

  confining (P = 0)               |P| = 0.0000  confining = True
    Z_2 acted: -0.0000+0.0000j   fixed by Z_2: True
    Z_3 acted: -0.0000+0.0000j   fixed by Z_3: True

  deconfining P = 1               |P| = 1.0000  confining = False
    Z_2 acted: -1.0000+0.0000j   fixed by Z_2: False
    Z_3 acted: -0.5000+0.8660j   fixed by Z_3: False

  deconfining P = 0.5+0.3j        |P| = 0.5831  confining = False
    Z_2 acted: -0.5000-0.3000j   fixed by Z_2: False
    Z_3 acted: -0.5098+0.2830j   fixed by Z_3: False



## 3. Svetitsky-Yaffe universality classes

Lean function `svetitskyYaffeClass`: SU(2) → 3D Ising, SU(3) → 3D 3-state Potts. Critical exponents (literature anchors):

- $\nu_{\rm Ising} = 0.6299$ (Pelissetto-Vicari 2002; Kos-Poland-Simmons-Duffin 2016 conformal bootstrap)
- $\nu_{\rm Potts} = 0.5$ (mean-field-like; the 3D 3-state Potts deconfinement is weakly first-order)

Strengthening pass replaced threshold-based `ising_nu_above_0_6` / `potts_nu_below_0_6` with the load-bearing direct comparison `ising_nu_gt_potts_nu`.

In [3]:
for N in [2, 3]:
    z = CenterZN(N)
    uc = svetitsky_yaffe_class(z)
    nu = critical_exponent_nu(uc)
    print(f'  SU({N}) → {uc.name:20s}   ν = {nu}')

nu_ising = critical_exponent_nu(UniversalityClass.ISING)
nu_potts = critical_exponent_nu(UniversalityClass.THREE_STATE_POTTS)
print()
print(f'  ν_Ising > ν_Potts:   {nu_ising > nu_potts}    ({nu_ising} > {nu_potts})')
print(f'  → mirrors Lean ising_nu_gt_potts_nu (load-bearing comparison, threshold-free).')

  SU(2) → ISING                  ν = 0.6299
  SU(3) → THREE_STATE_POTTS      ν = 0.5

  ν_Ising > ν_Potts:   True    (0.6299 > 0.5)
  → mirrors Lean ising_nu_gt_potts_nu (load-bearing comparison, threshold-free).


## 4. KSS bound: $\eta/s \geq 1/(4\pi) \in [0.07, 0.08]$

Lean theorems `KSS_bound_positive`, `KSS_bound_above_0_07`, `KSS_bound_below_0_08`. The bracket is anchored via Mathlib `Real.pi_gt_d4` (π > 3.141) and `Real.pi_lt_d4` (π < 3.142).

In [4]:
print(f'  KSS bound 1/(4π)            = {KSS_BOUND:.10f}')
print(f'  Lean bracket [0.07, 0.08]?    {0.07 < KSS_BOUND < 0.08}')
print(f'  KSS bound positive?          {KSS_BOUND > 0}')
print()
print(f'  Walker-Wang window [KSS, 2·KSS] = [{KSS_BOUND:.6f}, {2*KSS_BOUND:.6f}]')

  KSS bound 1/(4π)            = 0.0795774715
  Lean bracket [0.07, 0.08]?    True
  KSS bound positive?          True

  Walker-Wang window [KSS, 2·KSS] = [0.079577, 0.159155]


## 5. Walker-Wang transport correctness-push

Lean tracked-Prop `H_WalkerWangTransportNearKSS`: $\eta/s \in [\mathrm{KSS}, 2 \cdot \mathrm{KSS}]$. Two-conjunct (drop-conjunct test passes — both halves load-bearing).

- Witness: `walker_wang_witness_at_kss_lower` — $\eta/s = \mathrm{KSS}$ saturates the lower bound, satisfies both conjuncts.
- Falsifier 1 (`walker_wang_zero_eta_violator`): $\eta/s = 0$ violates KSS lower (uses `KSS_bound_positive`).
- Falsifier 2 (`walker_wang_unit_eta_violator`): $\eta/s = 1$ violates Walker-Wang upper (uses `KSS_bound_below_0_08` via $2 \cdot 0.08 = 0.16 < 1$).

In [5]:
for label, eta_over_s in [
    ('KSS lower (witness)',         KSS_BOUND),
    ('1.5 × KSS (within window)',   1.5 * KSS_BOUND),
    ('2 × KSS (upper bound)',       2 * KSS_BOUND),
    ('zero (falsifier — below KSS)', 0.0),
    ('unity (falsifier — above 2·KSS)', 1.0),
    ('3 × KSS (above window)',      3 * KSS_BOUND),
]:
    pred = WalkerWangPrediction(eta_over_s)
    print(f'  {label:38s}  η/s = {eta_over_s:.5f}   '
          f'above_kss = {pred.above_kss}   below_2kss = {pred.below_double_kss}   consistent = {pred.consistent}')

  KSS lower (witness)                     η/s = 0.07958   above_kss = True   below_2kss = True   consistent = True
  1.5 × KSS (within window)               η/s = 0.11937   above_kss = True   below_2kss = True   consistent = True
  2 × KSS (upper bound)                   η/s = 0.15915   above_kss = True   below_2kss = True   consistent = True
  zero (falsifier — below KSS)            η/s = 0.00000   above_kss = False   below_2kss = True   consistent = False
  unity (falsifier — above 2·KSS)         η/s = 1.00000   above_kss = True   below_2kss = False   consistent = False
  3 × KSS (above window)                  η/s = 0.23873   above_kss = True   below_2kss = False   consistent = False


## 6. Cross-bridges

**`higher_form_discrete_iff_non_abelian`** — genuine biconditional (NOT identity wrapper) linking discrete higher-form symmetry to non-Abelian gauge groups.

**`su3k1_fusion_card_matches_z3_order`** — calls `SKEFTHawking.su3k1_object_count` from `SU3kFusion.lean`: SU(3)$_1$ fusion category has $|{\rm SU}(3)_1\text{Obj}| = 3$, matching the order of $\mathbb{Z}_3$. The topological fusion ring categorifies the center.

In [6]:
Z3 = CenterZN(3)
print(f'  Order of Z_3 group               = {Z3.N}')
print(f'  |SU(3)_1 fusion objects|         = 3   (vac, fund, conj — Lean su3k1_object_count : Fintype.card SU3k1Obj = 3)')
print(f'  Match (categorification)         = {Z3.N == 3}')
print()
print('  Fusion rules of SU(3)_1 are isomorphic to Z_3 group addition:')
print('    fund ⊗ fund = conj   (1 + 1 = 2 mod 3)')
print('    fund ⊗ conj = vac    (1 + 2 = 0 mod 3)')
print('    conj ⊗ conj = fund   (2 + 2 = 1 mod 3)')

  Order of Z_3 group               = 3
  |SU(3)_1 fusion objects|         = 3   (vac, fund, conj — Lean su3k1_object_count : Fintype.card SU3k1Obj = 3)
  Match (categorification)         = True

  Fusion rules of SU(3)_1 are isomorphic to Z_3 group addition:
    fund ⊗ fund = conj   (1 + 1 = 2 mod 3)
    fund ⊗ conj = vac    (1 + 2 = 0 mod 3)
    conj ⊗ conj = fund   (2 + 2 = 1 mod 3)


## 7. Figure — Polyakov-loop scaling and Walker-Wang KSS window

**Left panel.** $|P|$ vs $T/T_c$ for SU(2) and SU(3) deconfinement, with universality-class scaling $|P| \propto (T/T_c - 1)^\beta$ above $T_c$ (β_Ising ≈ 0.326, β_Potts ≈ 0.111). Below $T_c$, $|P| = 0$ (confinement).

**Right panel.** Eight $\eta/s$ values on a log-scale bar chart: from 0 to 1, including KSS lower and Walker-Wang upper as horizontal reference lines. Bars inside the [KSS, 2·KSS] window are blue; outside (the two falsifiers and the trivial cases) are amber.

In [7]:
# viz-ref: fig_polyakov_loop_deconfinement
fig_polyakov_loop_deconfinement().show()

## 8. Lean theorem inventory (18 substantive)

All theorems in `CenterSymmetryConfinement.lean` are clean (zero `sorry`, zero new axioms; closure $\{\texttt{propext, Classical.choice, Quot.sound}\}$).

**§Center ℤ_N:** `centerPhase_pow_N`, `centerPhase_norm_one`, `centerPhase_Z2_eq_neg_one`, `centerAction_Z2_unit_eq_neg_one`.

**§Polyakov-loop / confinement:** `confining_invariant_under_center_action`, `nonzero_breaks_center_invariance`, `confining_iff_center_invariant`, `confining_iff_magnitude_zero`, `deconfining_implies_magnitude_positive`.

**§Svetitsky-Yaffe:** `ising_nu_gt_potts_nu` (load-bearing comparison; threshold-pair `ising_nu_above_0_6` and `potts_nu_below_0_6` REMOVED in strengthening pass).

**§KSS bound:** `KSS_bound_positive`, `KSS_bound_below_0_08`, `KSS_bound_above_0_07`.

**§Walker-Wang correctness-push:** `H_WalkerWangTransportNearKSS` (tracked Prop), `walker_wang_witness_at_kss_lower`, `walker_wang_zero_eta_violator`, `walker_wang_unit_eta_violator`.

**§Cross-bridges:** `higher_form_discrete_iff_non_abelian` (genuine biconditional), `su3k1_fusion_card_matches_z3_order` (calls `SU3kFusion.su3k1_object_count`).

Discipline trend: 6c.3 = 12 retroactive, 6b.1 = 5, **6d.1 = 6** (3 first-pass + 3 second-pass) — the second ruthless review caught patterns the first missed (identity-function wrapper, missing biconditional consolidation, missing existence witness). Multi-pass review until two consecutive clean passes is the safer protocol.